Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Chains

- A chain is a fixed pipeline ; we decide the path and it runs the same every time
- A sequential chain feeds one step's output straight into the next
- Nothing here decides anything at run time. That comes later, with agents

Builds a prompt-to-model chain, then a two-step chain that paraphrases and
then shortens.

### Exercise Simple chain (Prompt ➜ LLM)
Fill in the missing pieces to build the simplest chain that answers in Dutch.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
# TODO: Fill in the imports and the places marked ____
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
llm = make_llm()
prompt = ChatPromptTemplate.from_messages([
    ("system", "____"),
    ("user", "{question}")
])
chain = prompt | llm  | ____
# Expected result: a response in English
result = chain.invoke({"question": "In one sentence: what is a chargeback?"})
print(type(result), "\n", result)
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = make_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always answer in Dutch."),
    ("user", "{question}"),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "In one sentence: what is a chargeback?"})
print(type(result), "\n", result)

### Exercise Sequential chain
Create a two-step chain: (1) paraphrase, (2) shorten to 1 sentence.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
# TODO: Build a sequential chain prompt1 -> llm -> prompt2 -> llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = make_llm()


prompt1 = ChatPromptTemplate.from_messages([
    ("system", "Transform the user's statement into a more formal style."),
    ("user", "{text}")
])

prompt2 = ChatPromptTemplate.from_messages([
    ("system", "Summarize the previous text into a single sentence."),
    ("user", "{prev}")
])

# HINT: Use the | operator and key mapping with .map() or a simple lambda
stage1 = prompt1 | llm
# below, pass the result of stage1 into prompt2 (key 'prev'):
stage2_input_map = lambda x: {"prev": x.content if hasattr(x, "content") else str(x)}

chain = stage1 | stage2_input_map | (____ |____)

print(chain.invoke({"text": "LangChain lets you build complex flows with LLMs."}))
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = make_llm()

prompt1 = ChatPromptTemplate.from_messages([
    ("system", "Transform the user's statement into a more formal style."),
    ("user", "{text}"),
])

prompt2 = ChatPromptTemplate.from_messages([
    ("system", "Summarize the previous text into a single sentence."),
    ("user", "{prev}"),
])

# Stage 1: paraphrase into a formal style
stage1 = prompt1 | llm

# Map stage 1's AIMessage into stage 2's expected input key ("prev")
stage2_input_map = lambda x: {"prev": x.content if hasattr(x, "content") else str(x)}

# Full sequential chain: prompt1 -> llm -> map -> prompt2 -> llm
chain = stage1 | stage2_input_map | (prompt2 | llm)

print(chain.invoke({"text": "LangChain lets you build complex flows with LLMs."}))

### Try swapping the order

- Shorten first and paraphrase second, then compare with the original order
- Each step only sees what the step before it produced, so order changes the
  result even though the two prompts have not changed